# Product Recommendation System
In this notebook, we will build a Collaborative Filtering recommendation engine using the implicit feedback (interaction scores) generated in the data cleaning phase. We will compute user-user similarities to recommend products to a user based on what similar customers have purchased or viewed.

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [2]:
#load the processed datasets
products_df = pd.read_csv('../data/processed/cleaned_products.csv')
interactions_df = pd.read_csv('../data/processed/user_item_interactions.csv')

print(f"Loaded {len(interactions_df)} user-item interaction records.")
display(interactions_df.head())

Loaded 22116 user-item interaction records.


,user_id,product_id,view_count,purchase_count,interaction_score
0,1,1,1.0,0.0,1.0
1,1,2,1.0,0.0,1.0
2,1,15,2.0,0.0,2.0
3,1,16,3.0,0.0,3.0
4,1,18,1.0,0.0,1.0


## 1. Creating the User-Item Matrix
To perform collaborative filtering, we need to pivot our interaction data into a 2D matrix where rows represent users, columns represent products, and the values are the interaction scores. Missing interactions will be filled with 0.

In [3]:
#create the user-item interaction matrix
user_item_matrix = interactions_df.pivot(
    index='user_id', 
    columns='product_id', 
    values='interaction_score'
).fillna(0)

print(f"User-Item Matrix shape: {user_item_matrix.shape}")
display(user_item_matrix.head())

User-Item Matrix shape: (500, 200)


product_id,1,2,3,4,5,6,7,8,9,10,...,191,192,193,194,195,196,197,198,199,200
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,15.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


## 2. Building the Collaborative Filtering Model
We will use Cosine Similarity to measure the distance between users based on their interaction histories. Users with similar viewing and purchasing patterns will have a high similarity score (closer to 1).

In [4]:
#compute user-user cosine similarity
user_similarity_array = cosine_similarity(user_item_matrix)

#convert the array back into a dataframe for easier querying
user_similarity_df = pd.DataFrame(
    user_similarity_array, 
    index=user_item_matrix.index, 
    columns=user_item_matrix.index
)

print("User Similarity Matrix:")
display(user_similarity_df.iloc[:5, :5])

User Similarity Matrix:


user_id,1,2,3,4,5
user_id,,,,,
1,1.000000,0.024404,0.022822,0.060193,0.030222
2,0.024404,1.000000,0.038240,0.009500,0.097132
3,0.022822,0.038240,1.000000,0.216795,0.022726
4,0.060193,0.009500,0.216795,1.000000,0.053961
5,0.030222,0.097132,0.022726,0.053961,1.000000


## 3. Developing the Recommendation Engine
Now, we define a function that takes a `user_id` and outputs the top `N` recommended products. 
The logic is as follows:
1. Find the top `k` most similar users.
2. Identify products those similar users interacted with highly.
3. Filter out products the target user has already interacted with.
4. Rank the remaining products and return the top `N`.

In [5]:
def recommend_products(user_id, user_item_matrix, user_similarity_df, products_df, num_recommendations=3):
    #check if user exists in our matrix
    if user_id not in user_item_matrix.index:
        return "User not found. Try a cold-start recommendation (e.g., top selling products)."
    
    #get similar users (excluding the user themselves)
    similar_users = user_similarity_df[user_id].sort_values(ascending=False).drop(user_id)
    
    #isolate the top 10 most similar users
    top_similar_users = similar_users.head(10).index
    
    #get the items the target user has already interacted with
    target_user_history = user_item_matrix.loc[user_id]
    items_already_seen = target_user_history[target_user_history > 0].index.tolist()
    
    # aggregate scores for items interacted with by similar users
    recommendation_scores = {}
    
    for similar_user in top_similar_users:
        similarity_score = user_similarity_df.loc[user_id, similar_user]
        similar_user_history = user_item_matrix.loc[similar_user]
        
        #items the similar user interacted with
        items_interacted = similar_user_history[similar_user_history > 0].index
        
        for item in items_interacted:
            if item not in items_already_seen:
                if item not in recommendation_scores:
                    recommendation_scores[item] = 0
                #score = interaction_score * similarity_weight
                recommendation_scores[item] += similar_user_history[item] * similarity_score
                
    #sort the recommended items by accumulated score
    sorted_recommendations = sorted(recommendation_scores.items(), key=lambda x: x[1], reverse=True)
    
    #get the top N product IDs
    top_n_item_ids = [item_id for item_id, score in sorted_recommendations[:num_recommendations]]
    
    #merge with product dataframe to return readable results
    recommended_products = products_df[products_df['product_id'].isin(top_n_item_ids)][['product_id', 'product_name', 'category', 'brand']]
    
    return recommended_products

## 4. Testing the Recommendation System
Let's test the model using a sample user (e.g., User ID 102) to see if we get the expected output style mentioned in the business requirements.

In [6]:
#test the recommendation engine for user 102
test_user_id = 102
print(f"Recommendations for User: {test_user_id}")
recs = recommend_products(test_user_id, user_item_matrix, user_similarity_df, products_df, num_recommendations=3)

if isinstance(recs, pd.DataFrame):
    display(recs)
else:
    print(recs)

#let's test another user
test_user_id_2 = 24
print(f"\nRecommendations for User: {test_user_id_2}")
recs_2 = recommend_products(test_user_id_2, user_item_matrix, user_similarity_df, products_df, num_recommendations=3)
display(recs_2)

Recommendations for User: 102


,product_id,product_name,category,brand
7,8,Smartwatch,Electronics,Lenovo
21,22,Camera,Electronics,HP
139,140,Sofa,Home,Adidas



Recommendations for User: 24


,product_id,product_name,category,brand
17,18,Bluetooth Speaker,Electronics,Lenovo
46,47,Shorts,Clothing,Puma
94,95,Python Programming,Books,Sony


## 5. Serializing the Model Artifacts
To use this recommendation engine in our Streamlit dashboard (`app.py` and `inference.py`), we will export the `user_item_matrix` and the `user_similarity_df` as pickle files.

In [7]:
# save the matrices to the models directory
with open('../models/user_item_matrix.pkl', 'wb') as f:
    pickle.dump(user_item_matrix, f)

with open('../models/user_similarity_matrix.pkl', 'wb') as f:
    pickle.dump(user_similarity_df, f)

print("Collaborative Filtering model artifacts successfully saved to the models/ directory.")

Collaborative Filtering model artifacts successfully saved to the models/ directory.
